# পাঠ 09 - মেটাকগনিশন ডিজাইন প্যাটার্ন


## সেটআপ

এই নোটবুকটি Microsoft Agent Framework ব্যবহার করে Metacognition ডিজাইন প্যাটার্নটি প্রদর্শন করে।

**প্রয়োজনীয়তা:**
- Azure OpenAI deployment পরিবেশ ভেরিয়েবলগুলোর মাধ্যমে কনফিগার করা হয়েছে
- Azure CLI প্রমাণীকৃত (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## মেটাকগনিশন কী?

মেটাকগনিশন হল **চিন্তা সম্পর্কে চিন্তা করা**। এআই এজেন্টদের প্রসঙ্গে, এর মানে হল এমন এজেন্ট তৈরি করা যা করতে পারে:

- **স্ব-পর্যালোচনা করা** তাদের নিজস্ব আউটপুট ও যুক্তি প্রক্রিয়া সম্পর্কে
- **ভুল শনাক্ত করা** এবং নীরবে ব্যর্থ হওয়ার পরিবর্তে উপযুক্তভাবে পুনরুদ্ধার করা
- **মূল্যায়ন করা** যে তাদের উত্তরসমূহ পূর্ণ এবং সহায়ক কিনা
- **অভিযোজিত করা** তাদের কৌশল যখন প্রাথমিক পদ্ধতি কাজ না করলে (উদাহরণস্বরূপ, ব্যাকআপ সিস্টেমে ফিরে যাওয়া)

একটি মেটাকগনিটিভ এজেন্ট কেবল প্রশ্নের উত্তর দেয় না — এটি নিজেই তার কর্মক্ষমতা পর্যবেক্ষণ করে এবং তৎক্ষণাৎ সামঞ্জস্য করে।


## প্রাথমিক এবং ব্যাকআপ টুল

একটি সাধারণ মেটাকগনিশন প্যাটার্ন হল **বিকল্প কৌশল**। এজেন্ট প্রথমে একটি প্রাথমিক টুল চেষ্টা করে; যদি তা ব্যর্থ হয় (যেমন একটি 404 ত্রুটি), এজেন্ট ব্যর্থতা সনাক্ত করে এবং স্বচ্ছভাবে একটি ব্যাকআপ টুলে সুইচ করে।

এটি বাস্তব বিশ্বের সিস্টেমগুলির প্রতিফলন, যেখানে প্রাথমিক সার্ভিসগুলো অনুপলব্ধ থাকতে পারে এবং এজেন্টদের বিকল্প পথ বেছে নেওয়ার আগে সমস্যাটি নিজে-নিজে নির্ণয় করতে হয়।

নীচে আমরা দুটি ফ্লাইট লুকআপ টুল সংজ্ঞায়িত করছি:
- **Primary** — কভার করে প্যারিস, টোকিও, এবং বার্সেলোনা
- **Backup** — কভার করে বার্লিন, সিডনি, এবং নিউ ইয়র্ক সিটি


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ত্রুটি পুনরুদ্ধারের সঙ্গে স্ব-প্রতিফলিত এজেন্ট

নীচের এজেন্টকে নির্দেশ দেওয়া হয়েছে প্রথমে প্রধান ফ্লাইট সিস্টেমটি চেষ্টা করতে, ব্যর্থতা চিনতে, এবং স্বচ্ছভাবে ব্যাকআপ সিস্টেমে ফিরে যেতে। প্রতিটি প্রতিক্রিয়ার পরে এটি সংক্ষেপে নিজেই মূল্যায়ন করে যে এটি ব্যবহারকারীর প্রশ্নটি সম্পূর্ণরূপে উত্তর দিয়েছে কিনা।


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## স্ব-মূল্যায়ন প্যাটার্ন

মেটাকগনিশনের আরেকটি দিক হল **স্ব-মূল্যায়ন**: একটি পৃথক এজেন্ট (অথবা দ্বিতীয় ধাপে একই এজেন্ট) একটি প্রতিক্রিয়াকে সম্পূর্ণতা, সঠিকতা, এবং সহায়কতার দিক থেকে পর্যালোচনা করে।

নীচে আমরা একটি `ResponseEvaluator` এজেন্ট তৈরি করছি যা ভ্রমণ-এজেন্টের উত্তরগুলোকে তিনটি মাত্রায় স্কোর করবে।


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## সারাংশ

এই পাঠে আপনি শিখেছেন কিভাবে Microsoft Agent Framework ব্যবহার করে **মেটাকগনিটিভ এজেন্ট** তৈরি করতে:

- **স্ব-প্রতিফলন**: নিজেদের যুক্তি পর্যবেক্ষণ করে এবং কী ঘটেছিল তা স্বচ্ছভাবে জানানো এজেন্ট।
- **বিকল্পসহ ত্রুটি পুনরুদ্ধার**: একটি প্রাথমিক + ব্যাকআপ টুল প্যাটার্ন যেখানে এজেন্ট ব্যর্থতা সনাক্ত করে (উদাহরণস্বরূপ, 404 ত্রুটি) এবং স্বয়ংক্রিয়ভাবে বিকল্প উৎস চেষ্টা করে।
- **স্ব-মূল্যায়ন**: একটি পৃথক মূল্যায়নকারী এজেন্ট যা উত্তরগুলিকে সম্পূর্ণতা, সঠিকতা, এবং সহায়কতার জন্য স্কোর করে।

এই প্যাটার্নগুলো এজেন্টকে আরও দৃঢ়, স্বচ্ছ এবং বিশ্বাসযোগ্য করে তোলে — প্রোডাকশনে স্থাপন করার জন্য গুরুত্বপূর্ণ গুণাবলী।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকারোক্তি**:
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনুবাদ করা হয়েছে। যদিও আমরা যথার্থতার চেষ্টা করি, অনুগ্রহ করে জানুন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল ভাষায় থাকা মূল নথিটিকেই কর্তৃত্বপূর্ণ উৎস হিসেবে গণ্য করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ গ্রহণ করার পরামর্শ দেওয়া হয়। আমরা এই অনুবাদ ব্যবহারের ফলে সৃষ্ট কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
